# Debugging with notebook

In [2]:
import os
import logging
from dotenv import load_dotenv
from fastapi import FastAPI, HTTPException
from pydantic import BaseModel, Field
from langchain_google_genai import ChatGoogleGenerativeAI
from logging import getLogger
from langchain_core.output_parsers import PydanticOutputParser
from langchain_core.tools import tool

In [3]:

# Load environment variables from .env
load_dotenv()

# Configure logging
logging.basicConfig(format="%(asctime)s - %(name)s - %(levelname)s - %(message)s")
logging.getLogger().setLevel(logging.NOTSET)

logger = logging.getLogger(__name__)

# Access the values using os.environ
api_key = os.environ.get("GOOGLE_API_KEY")

app = FastAPI(title="Text Processing API")

# disable AFC
model = ChatGoogleGenerativeAI(
    model="gemini-3.6-flash",
    # stream_usage=True,
    # temperature=None,
    # max_tokens=None,
    # timeout=None,
    # reasoning_effort="low",
    # max_retries=2,
    api_key=api_key,  # If you prefer to pass api key in directly
    # base_url="...",
    # organization="...",
    # other params...
)

In [4]:

@tool
def add(a: int, b: int) -> int:
    """
    Add two numbers
    """
    return a + b +1

@tool
def multiply(a: int, b: int) -> int:
    """
    Multiply two numbers
    """
    return a * b

# bind tools to model
tools = [add, multiply]
model = model.bind_tools(tools)


# request and response models
class TextRequest(BaseModel):
    text: str

class TextResponse(BaseModel):
    translated_text: str = Field(description="Translated user sentence")
    is_formal: bool = Field(description="Whether the translated text is formal")

class CalculatedResponse(BaseModel):
    result: int = Field(description="The result of the calculation")
    answer_in_words: str = Field(description="The answer in words")


In [7]:
    messages = [
    (
        "system",
        "You are a helpful assistant that translates English to French. Translate the user sentence.",
    ),
    ("human", "Hi, Good morning!"),
    ]

    try:
        # structured output
        structured_model = model.with_structured_output(TextResponse, method="json_mode")
        output_parser = PydanticOutputParser(pydantic_object=TextResponse)
        format_instructions = output_parser.get_format_instructions()
        structured_response = structured_model.invoke(f"{messages} {format_instructions}")
        logger.debug(f"translate - request: {messages} {format_instructions}")

        logger.info("Successfully received response from LLM")
        logger.debug(f"translate - response: {structured_response}")
        # enforce the response model
        TextResponse(
            translated_text=structured_response.translated_text,
            is_formal=structured_response.is_formal
        )
    except Exception as e:
        logger.error(f"Error invoking LLM: {e}", exc_info=True)
        raise HTTPException(status_code=500, detail="Internal server error calling LLM")

2026-08-28 10:58:44,229 - google_genai.models - INFO - AFC is enabled with max remote calls: 10.
2026-08-28 10:58:44,230 - google_genai.models - WARNING - Direct use of automatic function calling (AFC) in Models.generate_content is not recommended. Instead, we recommend to use AFC in Chat.send_message. Similarly, direct use of AFC in Models.generate_content_stream is not recommended. Instead, we recommend to use AFC in Chat.send_message_stream.
2026-08-28 10:58:44,256 - httpcore.connection - DEBUG - connect_tcp.started host='generativelanguage.googleapis.com' port=443 local_address=None timeout=None socket_options=None
2026-08-28 10:58:44,360 - httpcore.connection - DEBUG - connect_tcp.complete return_value=<httpcore._backends.sync.SyncStream object at 0x000001E7BAA73770>
2026-08-28 10:58:44,361 - httpcore.connection - DEBUG - start_tls.started ssl_context=<ssl.SSLContext object at 0x000001E7BA8F8200> server_hostname='generativelanguage.googleapis.com' timeout=None
2026-08-28 10:58:44,

In [8]:
structured_model.invoke(f"{messages} {format_instructions}")

2026-08-28 11:00:36,813 - google_genai.models - INFO - AFC is enabled with max remote calls: 10.
2026-08-28 11:00:36,817 - httpcore.connection - DEBUG - close.started
2026-08-28 11:00:36,818 - httpcore.connection - DEBUG - close.complete
2026-08-28 11:00:36,819 - httpcore.connection - DEBUG - connect_tcp.started host='generativelanguage.googleapis.com' port=443 local_address=None timeout=None socket_options=None
2026-08-28 11:00:36,871 - httpcore.connection - DEBUG - connect_tcp.complete return_value=<httpcore._backends.sync.SyncStream object at 0x000001E7BAA9AE90>
2026-08-28 11:00:36,876 - httpcore.connection - DEBUG - start_tls.started ssl_context=<ssl.SSLContext object at 0x000001E7BA8F8200> server_hostname='generativelanguage.googleapis.com' timeout=None
2026-08-28 11:00:36,922 - httpcore.connection - DEBUG - start_tls.complete return_value=<httpcore._backends.sync.SyncStream object at 0x000001E7BAAA0180>
2026-08-28 11:00:36,934 - httpcore.http11 - DEBUG - send_request_headers.star

TextResponse(translated_text='Bonjour, bon matin !', is_formal=True)

In [19]:
messages = [
    (
        "system",
        "You are a helpful assistant that calculates numbers. Use the tools to answer the user's request.",
    ),
    ("human", "Add 111 and 222"),
    ]

answer = await model.ainvoke(f"{messages}")

2026-08-28 11:06:43,919 - httpcore.connection - DEBUG - connect_tcp.started host='generativelanguage.googleapis.com' port=443 local_address=None timeout=None socket_options=None
2026-08-28 11:06:44,221 - httpcore.connection - DEBUG - connect_tcp.complete return_value=<httpcore._backends.anyio.AnyIOStream object at 0x000001E7BACDC440>
2026-08-28 11:06:44,222 - httpcore.connection - DEBUG - start_tls.started ssl_context=<ssl.SSLContext object at 0x000001E7BA8F8200> server_hostname='generativelanguage.googleapis.com' timeout=None
2026-08-28 11:06:44,265 - httpcore.connection - DEBUG - start_tls.complete return_value=<httpcore._backends.anyio.AnyIOStream object at 0x000001E7BAA9B610>
2026-08-28 11:06:44,266 - httpcore.http11 - DEBUG - send_request_headers.started request=<Request [b'POST']>
2026-08-28 11:06:44,267 - httpcore.http11 - DEBUG - send_request_headers.complete
2026-08-28 11:06:44,268 - httpcore.http11 - DEBUG - send_request_body.started request=<Request [b'POST']>
2026-08-28 11:

GoogleRateLimitError: Error calling model 'gemini-3.6-flash' (RESOURCE_EXHAUSTED): 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. \n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 20, model: gemini-3.6-flash\nPlease retry in 41.004854005s.', 'status': 'RESOURCE_EXHAUSTED', 'details': [{'@type': 'type.googleapis.com/google.rpc.Help', 'links': [{'description': 'Learn more about Gemini API quotas', 'url': 'https://ai.google.dev/gemini-api/docs/rate-limits'}]}, {'@type': 'type.googleapis.com/google.rpc.QuotaFailure', 'violations': [{'quotaMetric': 'generativelanguage.googleapis.com/generate_content_free_tier_requests', 'quotaId': 'GenerateRequestsPerDayPerProjectPerModel-FreeTier', '

In [17]:
answer.text

''